# CreditLens MYPredict whether a loan applicant will default, turn that into a credit score, andexplain every decline.Data: Home Credit Default Risk (Kaggle). Features were built earlier by an AWS Gluejob and saved to S3 as Parquet.The shape follows the course template from Part 10, Section 49 (XGBoost):load, encode, split, train, evaluate. Everything after that is added one step at atime, and each step starts by showing why the simple version is not enough yet.

## 1. Libraries

In [ ]:
import jsonimport osimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltRANDOM_STATE = 42BUCKET = "creditlens-hazim-2026"

## 2. Load the data

In [ ]:
dataset = pd.read_parquet(f"s3://{BUCKET}/CreditLens/features/")print("rows:", f"{len(dataset):,}")print("columns:", dataset.shape[1])dataset.head()

## 3. Look at the target first`TARGET` is 1 if the applicant defaulted, 0 if they repaid.Check the balance before writing any model code. This one number decides how therest of the notebook has to work.

In [ ]:
print(dataset["TARGET"].value_counts())print()print("default rate:", round(dataset["TARGET"].mean() * 100, 2), "percent")

## 4. Remove features we are not allowed to useA model can find a pattern in anything. That does not mean it is legal to use it.Lending law in most countries, including the Equal Credit Opportunity Act and BNMguidelines, forbids deciding credit based on gender, marital status, or familysituation. If the model never sees those columns, it cannot base a decline on them.Hiding them from the reason list but leaving them in the model is not a fix. Thedecision would still be influenced by them, we just would not see it.One honest limitation: removing these columns does not make the model fair. Othercolumns can still stand in for them indirectly. This is the required first step, notthe finish line.

In [ ]:
PROTECTED = ["CODE_GENDER", "NAME_FAMILY_STATUS", "CNT_CHILDREN"]present = [c for c in PROTECTED if c in dataset.columns]dataset = dataset.drop(columns=present)print("removed:", present)print("columns left:", dataset.shape[1])

## 5. Split into X and y`SK_ID_CURR` is the applicant id. It is a label on the row, not information aboutthe person, so it goes too. Left in, the model can memorise individual applicants.

In [ ]:
y = dataset["TARGET"]X = dataset.drop(columns=["TARGET", "SK_ID_CURR"])print("features:", X.shape[1])

## 6. Encoding categorical dataThe course uses `LabelEncoder` and `OneHotEncoder`. LightGBM needs neither. Markinga column as `category` is enough and it handles the rest.`inf` shows up where a ratio divided by zero, for example debt divided by zeroincome. LightGBM accepts missing values but not infinity, so those become missing.

In [ ]:
X = X.replace([np.inf, -np.inf], np.nan)cat_cols = X.select_dtypes(include="object").columns.tolist()for col in cat_cols:    X[col] = X[col].astype("category")num_cols = X.select_dtypes(include=[np.number]).columns.tolist()print("categorical:", len(cat_cols), " numeric:", len(num_cols))

## 7. Splitting into three piles, not twoThe course splits into train and test. Here there are three:- **train**, the model learns from this- **valid**, used to correct the model's probabilities in step 11- **test**, never touched until the final scoreWhy not two: if the same data is used both to adjust the model and to grade it, thegrade is flattering rather than honest.`stratify=y` keeps the same 8 percent default rate in every pile. Without it arandom split can hand one pile a different mix and the numbers stop beingcomparable.

In [ ]:
from sklearn.model_selection import train_test_splitX_rest, X_test, y_rest, y_test = train_test_split(    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)X_train, X_valid, y_train, y_valid = train_test_split(    X_rest, y_rest, test_size=0.125, stratify=y_rest, random_state=RANDOM_STATE)for name, part in [("train", y_train), ("valid", y_valid), ("test", y_test)]:    print(f"{name:6s} {len(part):>7,} rows   default rate {part.mean() * 100:.2f} percent")

## 8. Baseline: logistic regressionAlways build the simplest model that could work before building a complicated one.If the complicated one cannot beat it, the complexity is not paying for itself.Numeric columns only, missing values filled with the median, everything scaled.`class_weight="balanced"` tells it to care about the rare defaults.

In [ ]:
from sklearn.pipeline import Pipelinefrom sklearn.impute import SimpleImputerfrom sklearn.preprocessing import StandardScalerfrom sklearn.linear_model import LogisticRegressionfrom sklearn.metrics import roc_auc_scorebaseline = Pipeline([    ("impute", SimpleImputer(strategy="median")),    ("scale", StandardScaler()),    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced",                               random_state=RANDOM_STATE)),])baseline.fit(X_train[num_cols], y_train)proba_baseline = baseline.predict_proba(X_test[num_cols])[:, 1]auc_baseline = roc_auc_score(y_test, proba_baseline)print("baseline AUC:", round(auc_baseline, 4))

## 9. Training LightGBMOut of 100 applicants, about 92 repay and 8 default. A model only trying to be rightas often as possible learns the lazy answer: say everyone repays, be right 92 percentof the time.`scale_pos_weight` makes each default count more during training, so ignoring themstops being the easy option. The value is just how many repayers there are perdefaulter.Keep this in mind for step 11: it fixes the learning, but it distorts theprobabilities the model reports.

In [ ]:
from lightgbm import LGBMClassifierimport lightgbm as lgbspw = (y_train == 0).sum() / (y_train == 1).sum()print("repayers per defaulter:", round(spw, 2))model = LGBMClassifier(    n_estimators=2000,    learning_rate=0.03,    num_leaves=31,    scale_pos_weight=spw,    random_state=RANDOM_STATE,)model.fit(    X_train, y_train,    eval_set=[(X_valid, y_valid)],    eval_metric="auc",    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(200)],)print("\nbest iteration:", model.best_iteration_)

## 10. Why accuracy lies hereThe course ends every classification section with `accuracy_score`. On balanced datathat is fine. Here it is actively misleading.Below, a "model" that predicts nobody ever defaults is scored the same way.It catches zero defaulters, which makes it worthless to a lender, and still scoresabove 90 percent.

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrixlazy = np.zeros(len(y_test))print("Confusion matrix, model that says nobody defaults:")print(confusion_matrix(y_test, lazy))print("\nAccuracy:", round(accuracy_score(y_test, lazy) * 100, 2), "percent")print("Defaulters found:", 0, "out of", int(y_test.sum()))

### AUC and Gini insteadAUC asks a better question: pick one defaulter and one repayer at random, how oftendoes the model give the defaulter the higher risk?- 0.50 means coin flip- 1.00 means perfect- 0.75 and above is good on this datasetThe lazy model scores 0.50, which is the honest verdict accuracy failed to give.Gini is `2 * AUC - 1`, the same information rescaled. Credit teams quote it out ofhabit.

In [ ]:
proba_raw = model.predict_proba(X_test)[:, 1]auc_lgb = roc_auc_score(y_test, proba_raw)print("lazy model AUC:  ", round(roc_auc_score(y_test, lazy), 4))print("baseline AUC:    ", round(auc_baseline, 4), " Gini:", round(2 * auc_baseline - 1, 4))print("LightGBM AUC:    ", round(auc_lgb, 4), " Gini:", round(2 * auc_lgb - 1, 4))

## 11. Fixing the probabilitiesStep 9 told the model to treat every default as if it were worth about 11 applicants.That was the right call for learning, but it means the probabilities it reports arepushed far too high. The cell below shows the size of the problem.This does not matter for AUC, which only cares about the order. It matters a lot forthe next step, because a credit score is built from the probability itself.The fix: `IsotonicRegression` learns the relationship between what the model predictsand what actually happened, using the valid pile. That pile was never trained on, sothe correction is learned from honest mistakes.

In [ ]:
from sklearn.isotonic import IsotonicRegressionproba_valid = model.predict_proba(X_valid)[:, 1]calibrator = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0)calibrator.fit(proba_valid, y_valid)proba = calibrator.predict(proba_raw)print("actual default rate:      ", round(y_test.mean(), 4))print("model says, before fix:   ", round(proba_raw.mean(), 4))print("model says, after fix:    ", round(proba.mean(), 4))print("\nAUC before:", round(roc_auc_score(y_test, proba_raw), 4),      " after:", round(roc_auc_score(y_test, proba), 4))

## 12. Turning probability into a credit scoreA probability of 0.13 means nothing to a loan officer. A score between 300 and 850does.The standard method is PDO, points to double the odds. Two settings control it:- `BASE_SCORE`, the score given to an average applicant- `PDO`, how many points it takes to halve the risk`BASE_ODDS` is taken from the data, not invented. It is how many people repay foreach one who defaults, so score 650 genuinely means "typical applicant here".

In [ ]:
BASE_SCORE = 650PDO = 40FLOOR, CAP = 300, 850BASE_ODDS = (1 - y_train.mean()) / y_train.mean()def probability_to_score(p_default):    p = np.clip(p_default, 1e-6, 1 - 1e-6)    odds = (1 - p) / p    factor = PDO / np.log(2)    offset = BASE_SCORE - factor * np.log(BASE_ODDS)    return np.clip(offset + factor * np.log(odds), FLOOR, CAP).round().astype(int)scores = probability_to_score(proba)print("base odds:", round(BASE_ODDS, 2), "to 1")print()print(pd.Series(scores).describe().round(1).to_string())

In [ ]:
plt.figure(figsize=(9, 4))plt.hist(scores[y_test.values == 0], bins=60, alpha=0.65, density=True, label="repaid")plt.hist(scores[y_test.values == 1], bins=60, alpha=0.65, density=True, label="defaulted")plt.xlabel("CreditLens score")plt.ylabel("share of applicants")plt.title("Score by what actually happened")plt.legend()plt.tight_layout()plt.show()

## 13. Choosing a cutoffThe model does not decide who gets a loan. The business does, by picking how manyapplicants it wants to approve. The model only says who is riskier than whom.The table below is the one a credit committee actually reads.

In [ ]:
TARGET_APPROVAL_RATE = 0.80result = pd.DataFrame({"score": scores, "actual": y_test.values})cutoff = int(np.percentile(result["score"], (1 - TARGET_APPROVAL_RATE) * 100))approved = result["score"] >= cutoffprint("cutoff:", cutoff)print("approved:", f"{approved.mean():.1%}", " bad rate:", f"{result.loc[approved, 'actual'].mean():.2%}")print("declined:", f"{(~approved).mean():.1%}", " bad rate:", f"{result.loc[~approved, 'actual'].mean():.2%}")bands = pd.cut(result["score"], bins=[300, 500, 550, 600, 650, 700, 850], include_lowest=True)print("\n", result.groupby(bands, observed=True).agg(    applicants=("actual", "size"),    bad_rate=("actual", lambda s: round(s.mean() * 100, 2)),).to_string())

## 14. Explaining a declineIf an applicant is declined they have a legal right to know why. A score alone isnot a reason.SHAP splits a single prediction into one number per feature, showing how much eachone pushed that person toward default.Two rules applied below:- Bureau scores like `EXT_SOURCE_2` are excluded from the letter. "Your credit score  is low" explains nothing and the applicant cannot act on it.- Everything else gets plain wording a person can actually read.

In [ ]:
import shapREASONS = {    "debt_to_income": "Total debt is high compared to income",    "credit_to_income": "Requested loan is large compared to income",    "annuity_to_income": "Monthly repayment is high compared to income",    "credit_utilization": "Existing credit lines are heavily used",    "bureau_max_overdue_days": "Recent overdue payments on record",    "bureau_active_loans": "Several loans are currently active",    "bureau_total_debt": "High total debt across all lenders",    "bureau_loan_count": "Large number of credit records",    "inst_late_count": "History of late instalment payments",    "inst_avg_days_late": "Instalments are usually paid late",    "inst_avg_pay_ratio": "Instalments are often underpaid",    "employed_years": "Short employment history",    "age_years": "Limited credit history length",}NOT_A_REASON = {"EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"}explainer = shap.TreeExplainer(model)X_sample = X_test.sample(2000, random_state=RANDOM_STATE)sv = explainer.shap_values(X_sample)if isinstance(sv, list):    sv = sv[1]sv = np.array(sv)if sv.ndim == 3:    sv = sv[:, :, 1]print("shap values:", sv.shape)

In [ ]:
def wording(name):    return REASONS.get(name, name.replace("_", " ").capitalize())def decline_reasons(shap_row, values, names, top_n=4):    out = []    for j in np.argsort(shap_row)[::-1]:        if shap_row[j] <= 0 or len(out) == top_n:            break        if names[j] in NOT_A_REASON:            continue        out.append((wording(names[j]), values.iloc[j], float(shap_row[j])))    return out# find someone who was actually declined, so the example is realsample_scores = probability_to_score(    calibrator.predict(model.predict_proba(X_sample)[:, 1]))i = int(np.where(sample_scores < cutoff)[0][0])print("Applicant:", X_sample.index[i])print("Score:", sample_scores[i], " Cutoff:", cutoff, " Decision: DECLINED")print("\nReasons:")for text, value, contribution in decline_reasons(sv[i], X_sample.iloc[i], X_sample.columns.tolist()):    print(f"  {text}")    print(f"     value {value}, weight {contribution:.3f}")

## 15. Which features mattered overallA sanity check that the model learned something sensible.

In [ ]:
importance = pd.Series(    model.booster_.feature_importance(importance_type="gain"), index=X.columns)top = importance.sort_values(ascending=False).head(15)plt.figure(figsize=(8, 5))top[::-1].plot(kind="barh")plt.title("Top 15 features by gain")plt.tight_layout()plt.show()

## 16. Save everything the app needs`app.py` in this folder loads these files and scores one applicant at a time.The calibrator and the scorecard settings are saved next to the model on purpose.If the app rebuilt a score using different settings, it would quietly disagree withthis notebook, and that is a very hard bug to notice.

In [ ]:
import joblibos.makedirs("artifacts", exist_ok=True)joblib.dump(model, "artifacts/creditlens_lgbm.joblib")joblib.dump(calibrator, "artifacts/creditlens_calibrator.joblib")demo = X_test.sample(500, random_state=RANDOM_STATE)demo.to_parquet("artifacts/sample_applicants.parquet")y_test.loc[demo.index].to_frame("TARGET").to_parquet("artifacts/sample_actuals.parquet")settings = {    "base_score": BASE_SCORE,    "base_odds": float(BASE_ODDS),    "pdo": PDO,    "score_floor": FLOOR,    "score_cap": CAP,    "cutoff": int(cutoff),    "target_approval_rate": TARGET_APPROVAL_RATE,    "scale_pos_weight": float(spw),    "calibration": "isotonic, fitted on the valid split",    "test_auc": float(auc_lgb),    "test_gini": float(2 * auc_lgb - 1),    "baseline_auc": float(auc_baseline),    "best_iteration": int(model.best_iteration_),    "n_features": int(X.shape[1]),    "removed_protected": present,    "adverse_action": REASONS,    "non_actionable_features": sorted(NOT_A_REASON),    "feature_names": X.columns.tolist(),    "column_order": X.columns.tolist(),    "categorical_levels": {c: [str(v) for v in X_train[c].cat.categories]                           for c in cat_cols},}with open("artifacts/creditlens_scorecard.json", "w") as f:    json.dump(settings, f, indent=2)print("saved:", sorted(os.listdir("artifacts")))

## Where this stopsWorking now: features from S3, a model measured honestly, a calibrated score, acutoff with its cost in defaults, and a readable reason for every decline.Not done yet, in order:1. Deploy the model as a SageMaker endpoint so it can be called over the internet2. Log incoming requests so drift can be checked later3. Write the README and publish the repo`app.py` covers the demo locally in the meantime, which is enough to show the wholething works end to end.